# Entity Resolution (MPI) Demo

`analytics_toolbox.entity_resolution` — link records for the same person across N source systems.

This notebook walks through the full pipeline:
1. Build a multi-system fixture (the same person under name variants across systems)
2. Configure the resolver
3. Resolve records into entity clusters
4. Read the cluster output
5. Blocking: DOB primary block + secondary fallback for null DOBs
6. Edge cases: name variants, true non-matches, missing fields
7. Tune `match_threshold`
8. HIPAA: the output holds only system IDs

**Privacy guarantee:** all processing is in-memory; no PHI appears in logs, errors, or output.

## 0. Setup

```bash
pip install analytics-toolbox
```

The `geospatial` extra is optional — if installed, street addresses are normalized with
scourgify before matching; if not, address normalization is skipped and every other field
still participates in scoring.

In [ ]:
import dataclasses
import textwrap
from pathlib import Path

import pandas as pd

from analytics_toolbox._config import load_config
from analytics_toolbox.entity_resolution import resolve
from analytics_toolbox.entity_resolution._config import EntityResolutionConfig
from analytics_toolbox.entity_resolution.fixtures.multi_system import make_mpi_fixture

print("Imports OK.")

## 1. A Multi-System Fixture

`make_mpi_fixture()` returns a `{id_column_name: DataFrame}` dict for five simulated systems.
The dict key is the name of each system's ID column — exactly the shape `resolve()` expects.

The data is deliberately messy, the way real source systems are:

- **John / Johny / Johnathan Smith** — the same person under three first-name variants,
  spread across systems A, B, and C. System C is also missing `Postal_Code`.
- **James / Jim Smooth** — a different person (systems D and E) whose `DOB` is null, so they
  fall back to secondary blocking. They share a last name with nobody in the Smith cluster.

In [ ]:
systems = make_mpi_fixture()

print("Systems:", list(systems.keys()))

# Stack all systems into one view to see the raw records side by side.
overview = pd.concat(
    [df.assign(system=name).rename(columns={name: "record_id"}) for name, df in systems.items()],
    ignore_index=True,
)
overview[["system", "record_id", "First_Name", "Last_Name", "DOB", "City", "Postal_Code"]]

## 2. Configure the Resolver

In practice you point `load_config()` at your `config.yaml`. Here we write a tiny config inline
so the notebook is self-contained. `match_threshold` is the one required setting — there is no
default, because the right cutoff is domain-specific.

In [ ]:
cfg_text = textwrap.dedent("""
    storage:
      connection: /tmp/er_demo.duckdb
      data_dir: /tmp/
    entity_resolution:
      match_threshold: 0.70
""")
cfg_path = Path("/tmp/er_demo_config.yaml")
cfg_path.write_text(cfg_text)

config = load_config(cfg_path)
print("match_threshold:", config.entity_resolution.match_threshold)
print("block_column:   ", config.entity_resolution.block_column)
print("secondary:      ", config.entity_resolution.secondary_block_columns)

## 3. Resolve

One call runs the whole pipeline: preprocess → block → score → cluster.

In [ ]:
clusters = resolve(systems, config=config)
clusters

## 4. Reading the Output

One row per linked entity. Each system column holds the matched record ID (or `None` if that
system had no record in the cluster), plus two confidence columns:

- `avg_similarity` — mean edge weight across the cluster's match graph
- `min_similarity` — the weakest link holding the cluster together

The three Smith records land in one row; the two Smooth records land in another. Records that
matched nothing above the threshold do not appear at all.

In [ ]:
smith = clusters[clusters["system_a_id"] == "456"].iloc[0]
print("Smith cluster:")
print(f"  system_a_id (John):      {smith['system_a_id']}")
print(f"  system_b_id (Johny):     {smith['system_b_id']}")
print(f"  system_c_id (Johnathan): {smith['system_c_id']}")
print(f"  avg / min similarity:    {smith['avg_similarity']:.3f} / {smith['min_similarity']:.3f}")
print(f"\nTotal clusters: {len(clusters)}")

## 5. Blocking — Why Null DOBs Still Match

Scoring every pair across every system would be O(N²). Blocking keeps comparisons local:

- **Primary block** — records are grouped by `DOB`. Only records sharing a DOB are compared.
- **Secondary block** — when `DOB` is null, the record falls back to
  `(Last_Name, Postal_Code)` so it can still find candidates.

The Smith records all share `DOB = 2025-01-06` (primary block). The Smooth records have a null
`DOB`, so they block together on `Last_Name = SMOOTH` instead — which is exactly why they link
to each other but never to the Smiths.

In [ ]:
dob_view = overview[["system", "record_id", "First_Name", "Last_Name", "DOB"]].copy()
dob_view["blocks_on"] = dob_view["DOB"].apply(
    lambda d: "DOB (primary)" if pd.notna(d) else "Last_Name+Postal (secondary)"
)
dob_view

## 6. Edge Cases

The fixture is built to exercise the behaviors that matter in an MPI:

| Case | Expectation |
|---|---|
| Name variants (John / Johny / Johnathan) | cluster **together** |
| Different people sharing nothing (Smith vs Smooth) | stay **separate** |
| System missing `Postal_Code` (Johnathan) | still participates; field excluded from its score |
| Null `DOB` (James / Jim) | secondary blocking links them |

In [ ]:
smith_rows = clusters[clusters["system_a_id"] == "456"]
smooth_rows = clusters[clusters["system_d_id"] == "124"]

# Name variants cluster together (3 Smith systems populated in one row).
assert smith_rows.iloc[0][["system_a_id", "system_b_id", "system_c_id"]].notna().all()

# Smith and Smooth never merge.
assert pd.isna(smith_rows.iloc[0]["system_d_id"])
assert pd.isna(smith_rows.iloc[0]["system_e_id"])

# System C linked despite having no Postal_Code.
assert smith_rows.iloc[0]["system_c_id"] == "1941"

smith_ids = list(smith_rows.iloc[0][["system_a_id", "system_b_id", "system_c_id"]])
smooth_ids = list(smooth_rows.iloc[0][["system_d_id", "system_e_id"]])
print("Smith cluster (system_a/b/c):", smith_ids)
print("Smooth cluster (system_d/e): ", smooth_ids)
print("\nAll edge-case expectations hold.")

## 7. Tuning `match_threshold`

The threshold is the minimum weighted similarity for a candidate pair to count as a link.
Lower it to catch looser matches (higher recall, more false positives); raise it to demand
near-exact agreement (higher precision, more missed links). Re-resolve at a stricter cutoff
and watch the looser links drop out.

In [ ]:
def resolve_at(threshold):
    cfg = dataclasses.replace(
        config,
        entity_resolution=EntityResolutionConfig(match_threshold=threshold),
    )
    return resolve(systems, config=cfg)

for t in (0.60, 0.70, 0.85, 0.95):
    result = resolve_at(t)
    linked = (result[["system_a_id", "system_b_id", "system_c_id",
                       "system_d_id", "system_e_id"]].notna().sum(axis=1) > 1).sum()
    print(f"threshold={t:.2f}  ->  {len(result)} clusters, {linked} multi-system")

## 8. HIPAA — The Output Is Safe to Share

The input DataFrames carry PHI (names, DOBs, addresses), but the cluster table does not: it
contains only system IDs and similarity scores. Nothing identifying ever reaches logs, error
messages, or the returned DataFrame — so the linkage result can be exported, inspected, or
logged without exposing patient data.

In [ ]:
phi_columns = {"First_Name", "Last_Name", "DOB", "Address", "City", "Postal_Code", "SSN"}
leaked = phi_columns & set(clusters.columns)

print("Output columns:", list(clusters.columns))
print("PHI columns in output:", leaked or "none")
assert not leaked, "PHI leaked into the cluster output"
print("\nClean: the cluster table holds only IDs and similarity scores.")